# 01 — Larsen public-source audit

The Introduction and Methods (week 6) is the governing specification. No employer records or employer-derived summaries are used. This notebook inspects the copied public replication assets without changing them. It produces audit and calibration-target summaries, **not a calibrated simulator or causal floor-response estimates**.

Run all cells from the `code-repo` directory. Dealer and fleet/lease records are summarized separately. Source identifiers are used only for internal duplicate checks; they are not exported as synthetic identifiers.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display
SOURCE_DIR = Path('Larsen_used_car_bargaining_data_and_code')
OUTPUT_DIR = Path('data/source_audit')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = {'dealer': 'pre_step1_ins0.csv', 'fleet_lease': 'pre_step1_ins1.csv'}
required_assets = list(FILES.values()) + ['Readme.pdf', 'step1_regression.do']
for name in required_assets:
    if not (SOURCE_DIR / name).is_file():
        raise FileNotFoundError(f'Missing source asset: {SOURCE_DIR / name}')


## Definitions and interpretation

The archive's `Readme.pdf` identifies `ins0` as dealers and `ins1` as fleet/lease. `idnew` identifies a run, not a longitudinal vehicle. `step1_regression.do` derives age from auction year minus model year and excludes `additional_obs == 1` for selected subsequent analyses. That exclusion is not automatically imposed on every summary here.

`low` is a secret reserve; `offeramt1` is the initial auction offer, and `finalprice` can reflect subsequent bargaining. Sales below reserve are counted, not discarded. The codebook describes zero prices for no-sales, but the CSV representation must be checked directly. A confirmed no-sale has zero gross proceeds and no conditional transaction price; an unknown outcome remains unknown.

Condition categories remain categorical, including “No grade.” Realized bids and sale fees are calibration/outcome variables, not predecision model predictors. Observed fees can inform a prospective fee assumption or schedule, but cannot simply be leaked into policy features.


In [2]:
field_map = pd.DataFrame([
    ('idnew', 'run identifier', 'Duplicate check only; not vehicle linkage'),
    ('history_run_date', 'auction date', 'Parse day-month-year; inspect coverage'),
    ('history_round_number', 'run sequence', 'Descriptive; does not establish vehicle linkage'),
    ('year', 'model year', 'Age = auction year minus model year'),
    ('make', 'vehicle make', 'Categorical; no invented segment mapping'),
    ('odo', 'mileage', 'Check missing and negative values'),
    ('conrepgrade', 'condition category', 'Preserve source categories'),
    ('auction', 'auction location', 'Preserve source categories'),
    ('low', 'reserve dollars', 'Divide by positive bb for floor ratio'),
    ('bb', 'benchmark dollars', 'Must be positive for normalized quantities'),
    ('agree', 'trade indicator', 'Only 0 and 1 are known outcomes'),
    ('offeramt1', 'initial auction offer dollars', 'Outcome, not predecision predictor'),
    ('finalprice', 'transaction price dollars', 'Conditional-price summaries use confirmed positive sales'),
    ('salefee', 'realized seller fee dollars', 'Summarize on eligible sales; not a known advance schedule'),
    ('additional_obs', 'supplementary regression observation', 'Record coverage; apply exclusions by purpose'),
], columns=['source_field', 'meaning', 'handling'])
display(field_map)
field_map.to_csv(OUTPUT_DIR / 'field_mapping.csv', index=False)


,source_field,meaning,handling
0,idnew,run identifier,Duplicate check only; not vehicle linkage
1,history_run_date,auction date,Parse day-month-year; inspect coverage
2,history_round_number,run sequence,Descriptive; does not establish vehicle linkage
3,year,model year,Age = auction year minus model year
4,make,vehicle make,Categorical; no invented segment mapping
5,odo,mileage,Check missing and negative values
6,conrepgrade,condition category,Preserve source categories
7,auction,auction location,Preserve source categories
8,low,reserve dollars,Divide by positive bb for floor ratio
9,bb,benchmark dollars,Must be positive for normalized quantities


## Load and audit each source separately

Counts below are diagnostic flags and may overlap. Each calibration summary uses a named eligibility mask; no universal complete-case deletion is applied. Nonnegative reserves are permitted in descriptive ratios, with zero reserves counted explicitly. Outlying ratios are retained at this stage so exclusion thresholds are not silently chosen.


In [3]:
sources = {}
manifest = []
audit_rows = []
eligibility_rows = []
for group, filename in FILES.items():
    path = SOURCE_DIR / filename
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    d = pd.read_csv(path, low_memory=False)
    missing_fields = set(field_map.source_field) - set(d.columns)
    if missing_fields:
        raise ValueError(f'{filename}: missing fields {missing_fields}')
    d['auction_date'] = pd.to_datetime(d.history_run_date, format='%d%b%Y', errors='coerce')
    for name in ['year', 'odo', 'low', 'bb', 'agree', 'finalprice', 'salefee', 'offeramt1']:
        d[name] = pd.to_numeric(d[name], errors='coerce')
    d['age'] = d.auction_date.dt.year - d.year
    known = d.agree.isin([0, 1])
    sold = d.agree.eq(1)
    unsold = d.agree.eq(0)
    benchmark_ok = d.bb.gt(0) & np.isfinite(d.bb)
    reserve_ok = d.low.ge(0) & np.isfinite(d.low)
    sale_price_ok = sold & d.finalprice.gt(0) & np.isfinite(d.finalprice)
    d['floor_ratio'] = (d.low / d.bb).where(benchmark_ok & reserve_ok)
    d['price_ratio'] = (d.finalprice / d.bb).where(benchmark_ok & sale_price_ok)
    d['transaction_price'] = d.finalprice.where(sale_price_ok)
    d['gross_proceeds'] = np.where(unsold, 0.0, d.transaction_price)
    d['sale_fee_for_summary'] = d.salefee.where(sale_price_ok & d.salefee.ge(0) & np.isfinite(d.salefee))
    d['below_reserve'] = (d.finalprice < d.low).where(sale_price_ok & reserve_ok)
    masks = {
        'known_outcome': known,
        'floor_ratio': benchmark_ok & reserve_ok,
        'conversion_with_floor_ratio': known & benchmark_ok & reserve_ok,
        'conditional_price_ratio': benchmark_ok & sale_price_ok,
        'sale_fee': d.sale_fee_for_summary.notna(),
        'below_reserve_comparison': sale_price_ok & reserve_ok,
    }
    audit_rows.append(dict(
        seller_group=group, rows=len(d), columns=len(pd.read_csv(path, nrows=0).columns),
        duplicate_run_ids=int(d.idnew.duplicated().sum()), missing_run_ids=int(d.idnew.isna().sum()),
        invalid_dates=int(d.auction_date.isna().sum()), start=str(d.auction_date.min().date()), end=str(d.auction_date.max().date()),
        negative_age=int(d.age.lt(0).sum()), missing_age=int(d.age.isna().sum()),
        negative_mileage=int(d.odo.lt(0).sum()), missing_mileage=int(d.odo.isna().sum()),
        missing_reserve=int(d.low.isna().sum()), negative_reserve=int(d.low.lt(0).sum()), zero_reserve=int(d.low.eq(0).sum()),
        invalid_benchmark=int((~benchmark_ok).sum()), unknown_outcome=int((~known).sum()),
        no_sales=int(unsold.sum()), no_sale_missing_price=int((unsold & d.finalprice.isna()).sum()),
        no_sale_zero_price=int((unsold & d.finalprice.eq(0)).sum()),
        no_sale_nonzero_price=int((unsold & d.finalprice.notna() & d.finalprice.ne(0)).sum()),
        sale_missing_or_invalid_price=int((sold & ~sale_price_ok).sum()),
        below_reserve_sales=int((sale_price_ok & reserve_ok & d.finalprice.lt(d.low)).sum()),
        conversion=float(d.loc[known, 'agree'].mean()),
        additional_observations=int(d.additional_obs.eq(1).sum()),
    ))
    for purpose, mask in masks.items():
        eligibility_rows.append(dict(seller_group=group, purpose=purpose, included=int(mask.sum()), excluded=int((~mask).sum())))
    sources[group] = d
    manifest.append(dict(seller_group=group, file=str(path), sha256=digest.hexdigest(), bytes=path.stat().st_size, rows=len(d)))
audit = pd.DataFrame(audit_rows)
eligibility = pd.DataFrame(eligibility_rows)
display(audit.set_index('seller_group').T)
display(eligibility)


seller_group,dealer,fleet_lease
rows,300740,211656
columns,79,79
duplicate_run_ids,0,0
missing_run_ids,0,0
invalid_dates,0,0
start,2006-11-07,2006-11-30
end,2010-03-09,2010-03-09
negative_age,0,0
missing_age,0,0
negative_mileage,0,0


,seller_group,purpose,included,excluded
0,dealer,known_outcome,300740,0
1,dealer,floor_ratio,281598,19142
2,dealer,conversion_with_floor_ratio,281598,19142
3,dealer,conditional_price_ratio,107404,193336
4,dealer,sale_fee,107404,193336
5,dealer,below_reserve_comparison,94170,206570
6,fleet_lease,known_outcome,211656,0
7,fleet_lease,floor_ratio,146477,65179
8,fleet_lease,conversion_with_floor_ratio,146477,65179
9,fleet_lease,conditional_price_ratio,152895,58761


## Calibration targets: distributions and dependence

These are descriptive source targets, not identified intervention effects. Numeric summaries use available valid values per field. Correlations use pairwise complete observations and include pair counts. Categorical proportions include missing values. Calendar coverage describes historical source auctions; a later simulation calendar must be explicitly documented.


In [4]:
quantile_rows, category_tables, correlation_rows = [], [], []
for group, d in sources.items():
    numeric = pd.DataFrame({
        'age': d.age.where(d.age.ge(0)), 'mileage': d.odo.where(d.odo.ge(0)),
        'benchmark': d.bb.where(d.bb.gt(0)), 'floor_ratio': d.floor_ratio,
        'price_ratio': d.price_ratio, 'sale_fee': d.sale_fee_for_summary,
    }).replace([np.inf, -np.inf], np.nan)
    for field in numeric:
        values = numeric[field].dropna()
        row = dict(seller_group=group, field=field, n=len(values), mean=values.mean(), sd=values.std())
        row.update({f'q{int(q*100):02}': values.quantile(q) for q in [.01,.05,.25,.5,.75,.95,.99]})
        quantile_rows.append(row)
    for field in ['make', 'conrepgrade', 'auction', 'history_round_number']:
        counts = d[field].fillna('<missing>').astype(str).value_counts(dropna=False).rename_axis('category').reset_index(name='count')
        counts['proportion'] = counts['count'] / len(d)
        counts['seller_group'], counts['field'] = group, field
        category_tables.append(counts)
    fields = ['age','mileage','benchmark','floor_ratio','price_ratio']
    for i, first in enumerate(fields):
        for second in fields[i+1:]:
            pair = numeric[[first, second]].dropna()
            correlation_rows.append(dict(seller_group=group, first=first, second=second, n=len(pair), correlation=pair[first].corr(pair[second])))
quantiles = pd.DataFrame(quantile_rows)
categories = pd.concat(category_tables, ignore_index=True)
correlations = pd.DataFrame(correlation_rows)
display(quantiles)
display(correlations)
display(categories[categories.field.eq('conrepgrade')])


,seller_group,field,n,mean,sd,q01,q05,q25,q50,q75,q95,q99
0,dealer,age,300740,6.515575,3.352714,1.000000,1.000000,4.000000,6.000000,9.000000,13.000000,14.000000
1,dealer,mileage,300740,96504.643872,46725.653435,12656.950000,26126.950000,62235.000000,93850.500000,126287.250000,177757.200000,222043.660000
2,dealer,benchmark,300740,7106.684262,4894.233792,700.000000,1350.000000,3275.000000,5900.000000,9825.000000,16925.000000,21525.000000
3,dealer,floor_ratio,281598,1.229703,0.585778,0.465116,0.666667,0.944056,1.116071,1.346154,2.125000,3.682914
4,dealer,price_ratio,107404,0.970996,0.416257,0.277778,0.441860,0.744681,0.932806,1.113924,1.611111,2.511628
5,dealer,sale_fee,107404,143.947972,54.961456,35.000000,60.000000,110.000000,135.000000,170.000000,249.250000,310.000000
6,fleet_lease,age,211656,3.467613,2.731788,0.000000,0.000000,1.000000,3.000000,5.000000,9.000000,11.000000
7,fleet_lease,mileage,211656,60856.631473,41923.740414,5996.550000,11023.750000,27377.500000,50627.000000,86719.000000,140512.000000,182532.300000
8,fleet_lease,benchmark,211656,10435.023104,6009.432735,1175.000000,2300.000000,5700.000000,9550.000000,14275.000000,21875.000000,26875.000000
9,fleet_lease,floor_ratio,146477,0.979913,0.301795,0.413462,0.642202,0.850242,0.947368,1.055344,1.400778,2.012270


,seller_group,first,second,n,correlation
0,dealer,age,mileage,300740,0.712585
1,dealer,age,benchmark,300740,-0.682169
2,dealer,age,floor_ratio,281598,0.163759
3,dealer,age,price_ratio,107404,-0.007409
4,dealer,mileage,benchmark,300740,-0.542146
5,dealer,mileage,floor_ratio,281598,0.124761
6,dealer,mileage,price_ratio,107404,-0.059948
7,dealer,benchmark,floor_ratio,281598,-0.304425
8,dealer,benchmark,price_ratio,107404,-0.107404
9,dealer,floor_ratio,price_ratio,94170,0.820938


,category,count,proportion,seller_group,field
42,No grade,292606,0.972953,dealer,conrepgrade
43,Average,4368,0.014524,dealer,conrepgrade
44,Clean,2310,0.007681,dealer,conrepgrade
45,Rough,1271,0.004226,dealer,conrepgrade
46,Extra Rough,102,0.000339,dealer,conrepgrade
47,Extra Clean,83,0.000276,dealer,conrepgrade
147,Average,94631,0.447098,fleet_lease,conrepgrade
148,Clean,48465,0.228980,fleet_lease,conrepgrade
149,Rough,37513,0.177236,fleet_lease,conrepgrade
150,No grade,21746,0.102742,fleet_lease,conrepgrade


## Save audit outputs and validate accounting

Exports contain aggregate summaries and file fingerprints, not copies of source records. Original assets remain unchanged. The field mapping is provisional until all calibration choices are finalized. Holding costs, retained inventory values, latent seller information, and counterfactual demand responses are not directly identified by these summaries.


In [5]:
for group, d in sources.items():
    assert d.loc[d.agree.eq(0), 'gross_proceeds'].eq(0).all()
    assert d.loc[d.agree.eq(0), 'transaction_price'].isna().all()
    assert d.loc[~d.agree.isin([0,1]), 'gross_proceeds'].isna().all()
    counts = eligibility[eligibility.seller_group.eq(group)]
    assert (counts.included + counts.excluded).eq(len(d)).all()
for name, table in {'audit_summary': audit, 'eligibility_counts': eligibility,
                    'numeric_targets': quantiles, 'category_targets': categories,
                    'correlation_targets': correlations}.items():
    table.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
(OUTPUT_DIR / 'source_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print('Audit completed. Outputs:', OUTPUT_DIR.resolve())


Audit completed. Outputs: /Users/ecilteodoro/Library/Mobile Documents/com~apple~CloudDocs/UWF/2026-8/IDC6940/code-repo/data/source_audit


## Next stage

Use the audited source targets to specify a joint synthetic generator, document the provenance of each parameter, and assess generated-versus-source distributions. Preserve bargaining interpretations: a binding-floor scenario cannot reproduce all below-reserve transactions by construction. Do not discard those sales merely to force the source data to match that scenario.

The existing `auction_project.ipynb` remains an uncalibrated demonstration and is not the capstone analysis. No calibrated simulation or policy conclusions are produced by this audit.
